# Can PCA trace Real Generating Pattern?

## Foundations

In this project, we treat matrices under financial settings, i.e. equities, which makes it easier to understand "signals" and "noises" that we would come up soon. Let‘s say: $A \in \mathbb{R}^{T \times N}$ is a return_matrix, with $T$ timing-observations (i.e daily returns, hours returns, etc) as rows, $N$ assests as columns. In practice, we centralize $A$:


$$
\bar{\mathbf{A}} = A - \mathbf{1}_T \frac{\mathbf{1}_T^\top A}{T}
$$


It induces the sample covariance matrix, which is:

$$
\mathbf{C} = \frac{1}{T} \bar{\mathbf{A}}^\top \bar{\mathbf{A}}
$$

We are in the regime where $T$ really large throughout the project, hence we put $T$ instead of $T-1$ here.


### Singular Value Decomposition (SVD)

Let $rk(A) = r$, we start from spectral-theorem on $A^\top A$, as it's a PSD matrix, we can write:

$$
A^\top A = P {S^2} P^\top
$$

By PSD and spectral theorem, we have $n$ non-negative eigenvalues along diagonal of $S^2$. $r$ of $n$ eigenvalues are $0$, the rest of $n-r$ eigenvalues are positve reals. By block-decomposition, we can reduce the spectral-diagonalization above as below:

$$
A^\top A = R {S*^2} R^\top
$$
Note:
- Here we pick $R_{n \times r}$ be first $r$ columns of $P$, $S*_{r \times r}$ be the digonal matrix with $r$ potive-real eigenvalues in descending order.
- By spectral, we know $P$'s columns are corresbouding orthonormal eigenvectors, hence $R^\top R = I_{r \times r}$.


$A$ is a centralized sample-matrix, we find its sample-covariance matrix $C$:
$$
C = \frac{1}{T} A^\top A = R \frac{S*^2}{T} R^\top
$$


Now, we construct SVD-decomposition:
$$
A = U S V^\top
$$

Note:
- $V$ is $R$ defined above, $S$ is $S*$ defined above. 
- $U_{T \times r} = A R S*^{-1}$, and $U^\top U = S*^{-1} R^\top A^\top A R S*^{-1} = I_{r \times r}$.


### Principal Component Analysis (PCA)

PCA is actually asking an optimization question: For a covariance matrix $\Sigma$, how to find $\displaystyle\max_{\|v\|=1} v^\top \Sigma v$, whicih we then call $v$ as the direction of largest variation (or direction of first principal component). To answer this question, we can use SVD-decomposition on $\Sigma$ to make everything clear!

For sample-covariance matrix $C$ above:
$$
C = R \frac{S*^2}{T} R^\top = P \frac{S^2}{T} P^\top
$$

We try to answer PCA's question above: As $R$'s column-vectors (orthonormal-eigenvectors of $A^\top A$) as principal components. By descending-ordered positve eigenvalues along $S^2$'s diagonal, and $P$'s columns spanning $\mathbb{R}^{n}$, pick arbitrary $\boldsymbol{v} = \sum_{i=1}^n \alpha_i \boldsymbol{p}_i, \quad \alpha_i \in \mathbb{R}, \quad \|v\|=1$:

$$
v^\top C v = v^\top P \frac{S^2}{T} P^\top v = \sum_{i=1}^n \alpha_i^2 \frac{S_{ii}^2}{T} \leq \frac{S_{11}^2}{T}
$$

The inequality holds when $\alpha_1 = 1, \quad \alpha_i = 0 \quad (i \neq 1)$, hence we find direction of largest variation (first principal component direction) is $p_{1}$, eigenvector(s) corresbouding to largest eigenvalue of sample-covariance matrix $C$.

People may ask: Why we care about a random optimization problem described above, say: $\displaystyle\max_{\|v\|=1} v^\top \Sigma v$, and calling it the direction of largest-vairiation? By idea of projection, we treat each row of $Av$ as the coefficient of projecting $A$'s row vector to direction of $sp\{v\}$. Variance of coefficients are equivalent to how far projected-data spread over this direction, which is how "information-intensive" this direction is. As $A$ is a centralized-matrix, we then know the coefficient-variance is exactly:
$$
\frac{(Av)^\top Av}{T} = v^\top \frac{1}{T} A^\top A v = v^\top C v
$$

So, we really find the direction of largest variation as above of $p_{1}$, and it's a direction of intensive-information, hence a good candiate of latent signal in market.


## Pure Noise Benchmark

$A$ is a pure-noise matrix with assumptions: Each entry of $A$ is an i.i.d Random Variable, with mean = 0, variance = $\sigma^2$. Then, $Marchenko-Pastur$ $Law$ for sample-covariance matrix of $A \in \mathbb{R}^{T \times N}$ states:

$$
C = \frac{1}{T} A^\top A
$$

$C$ has its spectrum mainly dropped between $[\lambda_{-}, \lambda_{+}]$, as $T$, $N$ goes to infinity, with $N/T$ goes to constant $c$:

$$
\lambda_{-} = \sigma^2\left(1 - \sqrt{\frac{N}{T}}\right)^2, \quad
\lambda_{+} = \sigma^2\left(1 + \sqrt{\frac{N}{T}}\right)^2
$$

In our pure-noise numerical experiment we take $\sigma=1$, giving the standard unit-variance MP edges.

Note:
- This is an asymptotic behaviour for high-dimensional random matrix, as $\frac{N}{T}$ kept constant, with N and T asymptotically goes to infnite.


## Low-rank Factor

Pure Noise Experiments indicate spectrum of sample covariance-matrix mostly fall inside $[\lambda_{-}, \lambda_{+}]$, we treat it as a benchmark. Now, we experiment by assuming low-rank factors exist in market's structure.

### One-factor model

$A$ is market-return matrix, we idealize market structure (real DGP) as:

$$
a_t = f_t * l + e_t 
$$

Assumptions:
- $a_t$ is transpose of return matrix $A$'s $t^{th}$ row.
- $f_t$ is an i.i.d R.V. as "factor" at time $t$.  $\mathbb{E}[f_t] = 0$, $\mathrm{Var}(f_t) = \tau^2$
- $l \in \mathbb{R}^{N \times 1}$ is fixed , and $\|\boldsymbol{l}\| = 1$
- $e_{ti}$ is an i.i.d R.V. as "noises for asset_i" at time $t$.  $\mathbb{E}[e_{ti}] = 0$, $\mathrm{Var}(e_{ti}) = \sigma^2$, $e_t$ is the tranpose of $E$'s $t^{th}$ row.
- $f_t$ is independent to $e_{ij}$


Extending to a time_periods $T$:
$$
A = f * l^\top + E
$$


As idealized structure doesn't vary by time $t$, we derive an underlying asset_return population covaraince martix:
$$
\Sigma = \mathbb{E}[(a_t - \mathbb{E}(a_t))(a_t - \mathbb{E}(a_t))^\top] = \mathbb{E}[a_t a_t^\top] = \mathbb{E}[f_t l l^\top f_t ^\top] + \mathbb{E}[e_t e_t^\top]
$$

By assumptions (independence and zero-mean) stated above, we have:
$$
\Sigma = l l^\top \mathbb{E}[f_t f_t^\top]  +  \sigma^2 * I_N = \tau^2 * l l^\top + \sigma^2 * I_N
$$

#### Appearance of a "spike"

Under Real Market - Structure (DGP), with population covariance matrix $\Sigma$ derived above: 

By $\|\boldsymbol{l}\| = 1$, we have:
$$
\Sigma l = \left(\tau^2 * l l^\top + \sigma^2 * I_N \right) l = \left(\tau^2 + \sigma^2 \right) l
$$

While, for any $v$ orthogonal to $l$:
$$
\Sigma v =  \left(\tau^2 * l l^\top + \sigma^2 * I_N \right) v = \sigma^2 v
$$

Reasonings above imply all eigenvalues of $\sum$ is:
$$
\sigma^2 + \tau^2, \quad \sigma^2,\quad \sigma^2,\quad ..., \quad\sigma^2
$$

By PCA, we easily know for $\Sigma$, the direction of largest variation (first principal component) is direction of $l$, which is exactly the loading-direction defined in one-factor model (real DGP).

#### Can sample covariance matrix recover real-structure ?

In population covariance matrix $\Sigma$, a "spike" eigenvalue $\sigma^2 + \tau^2$ with real-loading direction $l$ exists. In practice, we can only see sample return matrix $A$ generated by latent market structure (real DGP) and calculated sample-covariance matrix $C$. This will make eigenvalues spreading out instead of being fixed at $\tau^2$ and $\sigma^2 + \tau^2$. So, a natural question comes: For sample covariance matrix $C$, will the largest eigenvalue also be a "spike" like an outlier? Is first principal component (PC1) for $C$ a good indicator for real-loading direction $l$ just as $\Sigma$'s PC1 does?

$$
C = \frac{1}{T} A^\top A
$$

To research this question under "Random Matrix Theory", we have sample return matrix $A \in \mathbb{R}^{T \times N}$, and we let $T$, $N$ goes to infinity, with $N/T$ goes to a fixed constant. 

To find out how sample covariance matrix's spectrum looks like under the low-rank-factor model, we will cite several advanced theorems and conduct numerical experiments to varify them.

#### How strong factor needs to be to leave its spectral-signiture ?

By famous BBP Phase-transition [Baik–Ben Arous–Péché (2004)], it states:

$$q = \frac NT \quad \lambda_+=\sigma^2(1+\sqrt q)^2$$

$\sqrt q$ describes a phase-transition point, situations are completely different on each side:

Subcritical factor:
$$
\frac{\tau^2}{\sigma^2}
\le
\sqrt q
$$

In this case, majority of eigenvalues are kept in the MP-Bulk, and largest sample eigenvalue approaches $\lambda_+$, no outlier sample eigenvalues detected.

The leading principal component fails to recover the true loading direction, as: $|\langle\hat l,l\rangle|^2 \to0$.

Supercritical factor:
$$
\frac{\tau^2}{\sigma^2} >
\sqrt q
$$

An isolated outlier sample eigenvalue detaches from the MP-Bulk.

The leading principal component begins aligning with the true loading direction, as: $|\langle\hat l,l\rangle|^2 > 0 $


The idea of BBP Phase-transition: finite-rank perturbations do not alter the limiting empirical spectral distribution(i.e. in either cases above, the majority of sample eigenvalues are expected inside MP-Bulk), and they may create isolated outlier eigenvalues. And it further states how "strong" the factor needs to make largest eigenvalue being such a detached outlying eigenvalue.


### Multi-Factor Model

The one-factor model above is the simplest case in which a low-rank factor creates a spiked eigenvalue. In financial markets, we usually model the market structure (real DGP) by several latent factors. Now, let's take a look at k-factor model:

$$
a_t = f_{t1} * l_1 + f_{t2} * l_2 + \dots + f_{tk} * l_k + e_t
$$

Note (Assumptions):
- $a_t$ is transpose of return matrix $A$'s $t^{th}$ row.
- $f_t=(f_{t1},...,f_{tk})^\top$ is an i.i.d R.V. as "factor" at time $t$,  $\mathbb{E}[f_{ti}] = 0$, $\mathrm{Var}(f_{ti}) = \tau_i^2$.
- $l_i \in \mathbb{R}^{N \times 1}$ is fixed as column $i$ of $L$, $\|\boldsymbol{l_i}\| = 1$ and $L^\top L = I_k$
- $e_{ti}$ is an i.i.d R.V. as "noises for asset_i" at time $t$.  $\mathbb{E}[e_{ti}] = 0$, $\mathrm{Var}(e_{ti}) = \sigma^2$, $e_t$ is the tranpose of $E$'s $t^{th}$ row.
- $f_t$ is independent to $e_{t}$

By above details, we can express $a_t$ in matrix-form:
$$
a_t = L * f_t + e_t
$$

An important thing is: by above assumptions, "strength" of factor_i is $\tau_i^2$ distinct to factor_j's "strength" of $\tau_j^2$, implying a variation of "importance" between factors. Also, we make a strong assumption on real loading directions $L$, which we let $L^\top L = I_k$.

Now, let's find the population-covariance matrix $\Sigma$ of this k-factor model, by assumptions, we known $a_t$ is still i.i.d:
$$
\Sigma = \mathbb{E}[(a_t - \mathbb{E}(a_t))(a_t - \mathbb{E}(a_t))^\top] = \mathbb{E}[a_t a_t^\top] = \mathbb{E}[(L f_t + e_t)(L f_t + e_t)^\top] = \mathbb{E}[L f_t f_t^\top L^\top] + \mathbb{E}[e_t e_t^\top]
$$

$$
\Sigma = L \mathbb{E}[f_t f_t^\top] L^\top + \sigma^2 * I_N = L \begin{pmatrix} \tau_1^2 & & \\ & \ddots & \\ & & \tau_k^2 \end{pmatrix} L^\top + \sigma^2 * I_N = L \Sigma_f L^\top + \sigma^2 * I_N
$$

#### Appearance of k "spikes"

An orthogonal eigenvectors' subset for $\Sigma$ here is $l_1, l_2, \dots, l_k$ ( orthogonality came by our strong assumption of $L^\top L = I_k$). So, similar to one-factor model we discussed previously, here we have:


$$
\Sigma l_i = L \Sigma_f L^\top l_i + \sigma^2 l_i = (\tau_i^2 + \sigma^2) * l_i
$$

So, we have N eigenvalues in this model:
$$
\tau_1^2 + \sigma^2, \tau_2^2 + \sigma^2, \dots, \tau_k^2 + \sigma^2, \sigma^2, \dots, \sigma^2
$$

This shows the k spiked eigenvalues in the population covariance-matrix. Now, similar question as before: Will there be k spiked (outlier) eigenvalues for sample covariance matrix. Is it still related to the "strength"? Is first k PCs good candidates for recovery of k real loading directions as $l_1, \dots, l_k$?